# Solution D1 · Reading the training curves of XGBoost

**Type**: Interpret

## Objective

Read `evals_result()` of four configurations (depth 2, 4, 6 and 6 with `min_child_weight=20`) trained for 600 rounds without early stopping: where the validation log-loss minimum is, how large the training-validation gap is at the minimum and at the end, and which configuration is the best and why. Then
predict the early stopping round by looking at the curve.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(4000, n_features=12, n_informative=5, class_sep=0.8, flip_y=0.08, random_state=0)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)
configs = {"depth 2": dict(max_depth=2), "depth 4": dict(max_depth=4), "depth 6": dict(max_depth=6), "depth 6, min_child_weight 20": dict(max_depth=6, min_child_weight=20)}
curves = {}
for name, kw in configs.items():
    m = xgb.XGBClassifier(n_estimators=600, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", n_jobs=-1, random_state=0, **kw)
    m.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], verbose=False)
    curves[name] = m.evals_result()
print(list(curves))

## Commented solution

In [ ]:
table = pd.DataFrame(index=list(curves), columns=["min_round", "val_min", "train_at_min", "gap_at_min", "final_val", "final_gap"], dtype=float)
for name, r in curves.items():
    train, val = np.array(r["validation_0"]["logloss"]), np.array(r["validation_1"]["logloss"])
    k = int(val.argmin())
    table.loc[name] = [k + 1, val[k], train[k], val[k] - train[k], val[-1], val[-1] - train[-1]]
table.round(3)

In [ ]:
best = table["val_min"].idxmin()
most_overfit = table["final_gap"].idxmax()
degradation = table["final_val"] - table["val_min"]
es_round = {}
for name, r in curves.items():
    val = np.array(r["validation_1"]["logloss"])
    for t in range(1, len(val) + 1):
        if int(val[:t].argmin()) + 1 + 30 == t:                    # 30 rounds without improving on the minimum
            es_round[name] = t
            break
print("best:", best, "| most overfit:", most_overfit)
print("validation degradation if you do not stop:", degradation.round(3).to_dict())
es_cost = pd.Series({name: np.array(curves[name]["validation_1"]["logloss"])[:t].min() - table.loc[name, "val_min"] for name, t in es_round.items()})
print("early stopping rounds:", es_round)
print("cost of stopping at the first plateau (log-loss):", es_cost.round(4).to_dict())

## Check

In [ ]:
with verify():
    check(table.notna().all().all() and list(table.columns) == ["min_round", "val_min", "train_at_min", "gap_at_min", "final_val", "final_gap"], "complete table", "one row per configuration")
    check(table["min_round"].is_monotonic_decreasing is False and table.loc["depth 2", "min_round"] > table.loc["depth 6", "min_round"] + 100, "shallow trees reach the minimum much later (over 100 rounds after the deep ones)", "min_round")
    check(best in table.index and table.loc[best, "val_min"] == table["val_min"].min() and table["val_min"].max() - table["val_min"].min() < 0.05, "the four configurations have a similar validation minimum (difference below 0.05): depth matters less than the right round", "val_min")
    check(most_overfit == "depth 6" and table.loc["depth 6", "final_gap"] > 2 * table.loc["depth 2", "final_gap"], "depth 6 without constraints has the largest final gap (over double that of depth 2)", "final_gap")
    check(degradation["depth 6"] > 0.03 and degradation["depth 2"] < 0.01 and degradation["depth 6, min_child_weight 20"] < degradation["depth 6"] / 3, "without early stopping depth 6 loses over 0.03 of log-loss; depth 2 less than 0.01; min_child_weight cuts the effect to less than a third", "final_val - val_min")
    check(len(es_round) == 4 and all(30 < es_round[n] <= table.loc[n, "min_round"] + 30 for n in table.index), "the stop triggers at the first plateau: never after the global minimum plus 30 rounds, often before", "argmin(val[:t]) + 1 + 30 == t")
    check(es_cost is not None and (es_cost > -1e-9).all() and (es_cost < 0.01).all(), "stopping at the first plateau costs less than 0.01 of validation log-loss compared with the global minimum", "min(val[:es_round]) - val_min")
done()

## What you learned

- **The curves say three things:** where the validation minimum is (the right round), how large the training-validation gap is (the overfitting) and how the validation degrades after the minimum (the cost of not stopping). With `learning_rate=0.1` deep trees have the minimum early (round 116 with depth 6, 153 with 4) and shallow trees much later (566 with depth 2).
- **The minima are almost equal** (log-loss 0.307-0.334 for all four): depth changes *when* you reach the minimum more than *how low* it is. That is why the tuning of `max_depth` has a second-order effect if the early stopping is done well: the configuration with `min_child_weight=20` is the best, but by 0.005 over depth 4.
- **The final gap is a measure of overfitting** (depth 6: 0.36; depth 2: 0.15): it grows with depth and with the rounds. The cost of not stopping shows in the validation: after the minimum depth 6 loses 0.053 of log-loss, depth 2 only 0.002.
- **`min_child_weight` is a strong regularization:** with depth 6 and `min_child_weight=20` the degradation after the minimum drops from 0.053 to 0.008 and the minimum moves from 116 to 265 rounds: it prevents isolating a few examples, which is the road of deep boosting to memorization.
- **Early stopping halts at the first plateau, not at the global minimum:** with patience 30 depth 2 stops at 470 rounds, before the global minimum (566), and the others at 146-199 rounds. The cost in log-loss is negligible (less than 0.01): the curve is flat around the minimum.

## Common pitfall

Comparing configurations after a **fixed** number of rounds (for example 200 for all): the deep ones are already past the minimum and look worse only because you did not stop them. Compare each one at its own best round.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How do you read the training curve of a boosting model?</summary>

The training log-loss always drops; the validation one drops, touches a minimum and then climbs back. The minimum is the right number of rounds, the gap between the two curves measures the overfitting, and the climb back is the cost of not stopping. Configurations are compared each at its own minimum (early stopping),
not at a fixed number of rounds.

</details>